# 09 — Species-Metabolite Pathway Correlation Network (v2)

**Last notebook in the pipeline.** Reads from NB02 (`preprocessed_data_reduced_v2.pkl`, `YACHIDA-CRC-2019_polyamine_partial_correlations_v2.csv`),
NB03 (`ml_results_v2.pkl`), NB06 (`mofa_factor_loadings_v2.csv` / `mofa_factor_scores_v2.csv`),
and NB08 (`r1_agora2_producer_enrichment_v2.csv`, `r8_polyamine_dicarb_overlap_v2.csv`).

| Module | Analysis | Data |
|--------|----------|------|
| D | GLASSO partial-correlation species-metabolite network, **with a permutation null model** | `preprocessed_data_reduced_v2.pkl` |
| B | Species-level AGORA2 enzyme-profile evidence for producer species | AGORA2 GENRE reconstructions, `ml_results_v2.pkl` |
| C | Curated pathway-keyword mapping (honestly labeled, not a live API) + species-pathway correlations | `preprocessed_data_reduced_v2.pkl` |
| G | AGORA2-EC-defined guild scores x MOFA+ factors, stage-stratified figure | `mofa_factor_scores_v2.csv`, AGORA2 |
| **Final synthesis** | **Cross-notebook evidence triangulation** on a SHAP-independent candidate pool (NB02 partial correlations) with NB03 SHAP, AGORA2 and NB06 MOFA as separate evidence columns | `YACHIDA-CRC-2019_polyamine_partial_correlations_v2.csv`, `ml_results_v2.pkl`, AGORA2, `mofa_factor_loadings_v2.csv` |

## Fixes applied vs. `exploratory/09_pathway_network.ipynb`

1. **Forward-reference dict pattern eliminated.** The old notebook's Module A (`cell 9`)
   referenced `METACYC_PW_ENZYME_FALLBACK` / `POLYAMINE_ENZYME_FAMILIES` inside a loop, but
   defined both dicts *after* that loop, in the same cell — any live-API-miss branch that
   needed the fallback raised `NameError`. Every dict/helper in this notebook is defined
   before the code that uses it, in every cell, with no exception. Module A itself (live
   MetaCyc REST + this fallback machinery) is dropped rather than ported forward — see
   Module B markdown below for why.
2. **Module B's genus-to-KEGG-organism-code matching is replaced with the shared AGORA2
   evidence** (`utils.build_agora2_ec_index()` + `utils.species_encodes_enzyme()`),
   consistent with NB08. The old approach compared a 4-character genus-name prefix against
   a same-or-shorter-length KEGG organism code (`genus[:4] in org.lower()`) — a substring
   containment test between two strings of essentially unrelated origin, which is true
   almost never, and when true is coincidental, not biological evidence. It also depended
   on live, uncached `rest.kegg.jp` calls (B1/B2/B3 sub-modules querying KEGG/NCBI/UniProt),
   which are dropped entirely along with it.
3. **Pathway mapping is honestly labeled.** The old notebook defined a `reactome_query()`
   function implying a live Reactome REST call, but the actual mapping used elsewhere in
   the same module was a small hardcoded keyword-substring dict (`REACTOME_PATHWAY_MAP`)
   that `reactome_query()` was never actually plugged into. Module C below keeps only the
   curated dict, clearly labeled as a manually-curated reference (the R-HSA identifiers are
   real Reactome pathway IDs, looked up by hand — not fetched live), and drops the unused,
   misleadingly-named live-query function.
4. **Every `pa_producers`-empty crash is guarded properly.** The old notebook printed a
   warning when `pa_producers` was missing/empty but then still indexed into it a few cells
   later (Module A/B `for _, row in pa_producers.iterrows()` etc.) and crashed with
   `KeyError`/`IndexError`. Every cell here that depends on `pa_producers` checks
   `HAS_PA_PRODUCERS` up front and prints a clear skip message — it does not just warn and
   then crash anyway.
5. **One missing-data policy, used consistently.** The old notebook used MICE in Module D,
   implicit skip-NaN in Module C (`spearmanr` fed raw un-imputed values), and zero-fill in
   Module G (`spc_aligned2[sp].fillna(0)`) — three different policies with no stated reason
   for the difference. This notebook picks **MICE** (`IterativeImputer`, via the single
   `mice_impute()` helper defined once below) as the one default and reuses the same
   imputed species/metabolite matrices in Modules D, C, and G.
6. **GLASSO edges now carry a permutation-based null-model check.** The old notebook
   selected edges by a bare `|partial_corr| >= 0.08` magnitude threshold with zero
   statistical validation — unlike the FDR-corrected correlations used everywhere else in
   this notebook and pipeline. Module D below builds a permutation null (shuffling the
   metabolite-block sample correspondence and refitting GraphicalLasso at the
   already-selected alpha) and reports BH-FDR-corrected empirical p-values per edge. If too
   few permutations converge to support this, the edges are saved with an explicit
   `Validated=False` caveat column instead of a silent, unvalidated magnitude cut.
7. **Genuinely consumes NB08's AGORA2 output.** The old notebook never read anything from
   NB07/NB08 despite the pipeline's implied ordering, redefining its own pathway universe
   from scratch. This notebook loads `r1_agora2_producer_enrichment_v2.csv` (used in Module
   B's cross-check and as a synthesis source) and `r8_polyamine_dicarb_overlap_v2.csv`
   (cross-referenced against Module C's significant species) directly.
8. **No hardcoded absolute paths.** The old notebook hardcoded
   `RESULTS_DIR = Path('c:/Users/andna/Documents/KI/Results')`. This notebook imports
   `utils` (`sys.path.insert(0, '.')`, run from `v2/`) and uses
   `utils.RESULTS_DIR` exclusively.
9. **Stage stratification uses `utils.STAGE_GROUP_MAP` / `utils.STAGE_GROUP_ORDER`.** The
   old Module G boxplot hand-duplicated a local `stage_order` / `stage_pal` using the
   *fine-grained*, pre-fix ordering (`Healthy, Stage_0, HS, Stage_I_II, Stage_III_IV, MP`),
   repeating the same broken MP/HS placement documented as wrong in `utils.py`'s
   `STAGE ENCODING` block. The final stage-stratified figure here uses the corrected,
   grouped `Stage.Group` / `STAGE_GROUP_ORDER` (`Healthy, Adenoma, Early_CRC,
   Advanced_CRC`; HS excluded) throughout, with no local re-derivation.
10. **Missing upstream inputs fail loudly.** A contract-required input (MOFA tables, NB08's
    r1/r8, NB02's partial correlations) that is missing raises `FileNotFoundError` instead of
    silently skipping a module (set `V2_ALLOW_MISSING=1` to downgrade to a warning).
11. **Module D is no longer empty.** GraphicalLassoCV on all 656 features (p > n) chose an
    alpha so large that no edge survived; the network is now fitted on ~110 features
    (top-variance species, partial-correlation species, polyamines, top-variance
    metabolites) with a 10-value alpha grid. Edge significance uses a Westfall-Young
    max-statistic permutation p-value (family-wise error over all cross edges), because
    per-edge BH over thousands of edges cannot reach q < 0.05 with 100 permutations.
12. **Module C matches pathways by KEGG ID** (whole-word name matching only as fallback):
    substring matching assigned hippurate to "urate" and hydroxyurea to "urea".
13. **Module G and the final synthesis are no longer circular.** Module G's guilds were
    genus lists copied from NB03's SHAP top genera, and the synthesis candidate pairs came
    from the SHAP list itself (so the SHAP source could only ever "support"). Guilds are now
    defined by AGORA2 EC numbers, and candidates are NB02's partial-correlation-significant
    species-polyamine pairs, with SHAP as one independent evidence column.

## New: relocated cross-notebook evidence synthesis

The old `03_ML.ipynb` had a late cell ("Producer Evidence Matrix") that tried to read
output files from NB06/NB08/NB09 — notebooks that run **after** NB03 in pipeline order — so
on any fresh sequential run those files simply didn't exist yet; it only ever produced
non-trivial numbers because of stale leftover files from an unrelated prior run. `v2`'s
`03_ml_benchmarking.ipynb` explicitly stops after producing its own outputs and documents
that this step moves here. The final section scores every NB02 partial-correlation-significant,
positively associated species-polyamine pair against three independent sources (NB03 SHAP,
AGORA2 enzyme encoding, NB06 MOFA co-loading) and reports real coverage numbers rather than
silently treating a missing source as "no evidence found".


In [ ]:
import sys
sys.path.insert(0, '.')

import os
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import spearmanr, kruskal
from sklearn.covariance import GraphicalLasso, GraphicalLassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer  # noqa: F401 (must precede IterativeImputer import)
from sklearn.impute import IterativeImputer
from statsmodels.stats.multitest import multipletests

import utils

RESULTS_DIR = utils.RESULTS_DIR
TABLES      = RESULTS_DIR / 'tables'
FIGURES     = RESULTS_DIR / 'figures' / 'pathway_network'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

YACHIDA = 'YACHIDA-CRC-2019'

# ── NB02 output (required) ──────────────────────────────────────────────────
with open(RESULTS_DIR / 'intermediate' / 'preprocessed_data_reduced_v2.pkl', 'rb') as f:
    pdata = pickle.load(f)

spc_df  = pdata['transformed_species'][YACHIDA].copy()
mtb_df  = pdata['transformed_mtb'][YACHIDA].copy()
meta_df = pdata['harmonized_meta'][YACHIDA].copy()
polyamine_columns = pdata.get('polyamine_columns', {}).get(YACHIDA, {})   # {kegg_id: column_name}
mtb_kegg_map      = pdata.get('mtb_kegg_map', {}).get(YACHIDA, {})        # {column_name: kegg_id}

ALLOW_MISSING = bool(os.environ.get('V2_ALLOW_MISSING'))

def _require(path, producer):
    """Contract-required input: fail loudly unless V2_ALLOW_MISSING is set."""
    if path.exists():
        return True
    msg = f"{path.name} not found -- run {producer} first"
    if ALLOW_MISSING:
        print('!!! WARNING: ' + msg + ' (V2_ALLOW_MISSING set: dependent steps will be skipped)')
        warnings.warn(msg)
        return False
    raise FileNotFoundError(msg + ' (or set V2_ALLOW_MISSING=1 to skip dependent steps)')

# ── NB02 partial correlations (required): SHAP-independent candidate pool ──
pc_path = RESULTS_DIR / 'tables' / 'YACHIDA-CRC-2019_polyamine_partial_correlations_v2.csv'
_require(pc_path, '02_analysis.ipynb')
partial_corr = pd.read_csv(pc_path)
print(f"NB02 partial correlations: {len(partial_corr)} species-polyamine pairs, "
      f"{int((partial_corr['QValue_partial'] < utils.FDR_THRESHOLD).sum())} with q<{utils.FDR_THRESHOLD}")

# ── NB03 output (required) ──────────────────────────────────────────────────
with open(RESULTS_DIR / 'intermediate' / 'ml_results_v2.pkl', 'rb') as f:
    ml_results = pickle.load(f)

pa_producers_all = ml_results.get('pa_producers', pd.DataFrame(columns=['Species', 'Polyamine', 'Rank']))
oof_shap_store   = ml_results.get('oof_shap_store', {})
# Producers = positive-direction SHAP species only (NB03 'Direction').
if 'Direction' in pa_producers_all.columns:
    pa_producers = pa_producers_all[pa_producers_all['Direction'] == 'positive'].copy()
else:
    pa_producers = pa_producers_all.copy()
HAS_PA_PRODUCERS = pa_producers is not None and not pa_producers.empty
if not HAS_PA_PRODUCERS:
    print("!" * 80)
    print("!!! WARNING: ml_results_v2.pkl['pa_producers'] has no positive-direction rows --")
    print("!!! Module B and the SHAP source of the final synthesis will be SKIPPED.")
    print("!" * 80)
    warnings.warn('pa_producers empty')

# ── NB06 outputs (loaded defensively — NB06 may not have run yet) ──────────
mofa_scores_path   = TABLES / 'mofa_factor_scores_v2.csv'
mofa_loadings_path = TABLES / 'mofa_factor_loadings_v2.csv'

if _require(mofa_scores_path, '06_multiomics_factors.ipynb'):
    mofa_scores = pd.read_csv(mofa_scores_path)
    HAS_MOFA_SCORES = True
else:
    print(f"WARNING: {mofa_scores_path.name} not found — Module G's MOFA-factor "
          f"correlation step will be SKIPPED.")
    mofa_scores = pd.DataFrame(columns=['Sample'])
    HAS_MOFA_SCORES = False

if _require(mofa_loadings_path, '06_multiomics_factors.ipynb'):
    mofa_loadings = pd.read_csv(mofa_loadings_path)
    HAS_MOFA_LOADINGS = True
else:
    print(f"WARNING: {mofa_loadings_path.name} not found — the final evidence synthesis's "
          f"MOFA co-loading source (Source C) will be treated as UNAVAILABLE for every pair.")
    mofa_loadings = pd.DataFrame(columns=['Factor', 'View', 'Feature', 'Weight'])
    HAS_MOFA_LOADINGS = False

# ── NB08 outputs (loaded defensively — NB08 may not have run yet) ──────────
r1_path = TABLES / 'r1_agora2_producer_enrichment_v2.csv'
r8_path = TABLES / 'r8_polyamine_dicarb_overlap_v2.csv'

if _require(r1_path, '08_pathway_analysis.ipynb'):
    r1_df = pd.read_csv(r1_path)
    HAS_R1 = True
else:
    print(f"WARNING: {r1_path.name} not found — Module B's cross-check against NB08 and "
          f"the final evidence synthesis's AGORA2 source (Source B) will be treated as "
          f"UNAVAILABLE.")
    r1_df = pd.DataFrame(columns=['Species', 'Polyamine', 'Rank', 'KEGG_ID',
                                   'Encodes_Enzyme', 'AGORA2_Model_Available'])
    HAS_R1 = False

if _require(r8_path, '08_pathway_analysis.ipynb'):
    r8_df = pd.read_csv(r8_path)
    HAS_R8 = True
else:
    print(f"WARNING: {r8_path.name} not found — Module C's cross-reference against NB08's "
          f"polyamine/dicarboxylic-acid overlap will be SKIPPED.")
    r8_df = pd.DataFrame(columns=['Species', 'Polyamine', 'KEGG_ID', 'Rho_polyamine',
                                   'QValue_polyamine', 'Dicarb_Metabolite',
                                   'Dicarb_KEGG_ID', 'Rho_dicarb', 'QValue_dicarb'])
    HAS_R8 = False

print(f"\nLoaded: spc={spc_df.shape}, mtb={mtb_df.shape}, meta={meta_df.shape}")
print(f"pa_producers: {pa_producers.shape} (HAS_PA_PRODUCERS={HAS_PA_PRODUCERS})")
print(f"mofa_scores: {mofa_scores.shape} (HAS_MOFA_SCORES={HAS_MOFA_SCORES}) | "
      f"mofa_loadings: {mofa_loadings.shape} (HAS_MOFA_LOADINGS={HAS_MOFA_LOADINGS})")
print(f"r1_agora2_producer_enrichment: {r1_df.shape} (HAS_R1={HAS_R1}) | "
      f"r8_polyamine_dicarb_overlap: {r8_df.shape} (HAS_R8={HAS_R8})")


## Missing-data policy (single default, documented once)

Every module below that needs a complete numeric matrix (Module D's GLASSO fit, Module C's
species-pathway correlations, Module G's enrichment scores) uses **MICE**
(`sklearn.impute.IterativeImputer`) via the one `mice_impute()` helper defined in the next
cell. No module falls back to a different policy (skip-NaN, zero-fill, mean-fill) — the old
notebook did all three, in three different modules, with no stated reason for the
difference. `mice_impute()` is a genuinely-missing-value policy; it is not used for
structurally-absent data (e.g. a sample simply not present in the MOFA+ fit), which is
handled by restricting to the samples that are actually present instead of imputing a
value for them.


In [ ]:
def mice_impute(mat_df, max_iter=10, random_state=42):
    '''The single missing-data policy for this notebook: MICE via IterativeImputer.

    `mat_df` is a numeric DataFrame indexed by Sample (no 'Sample' column). Returns a
    same-shaped, same-indexed DataFrame with every remaining NaN imputed. Reused by
    Module D (GLASSO), Module C (species-pathway correlations), and Module G (pathway
    enrichment scores) so all three use the same policy on the same underlying values.
    '''
    if not mat_df.isna().any().any():
        return mat_df.copy()          # nothing to impute
    imputer = IterativeImputer(max_iter=max_iter, random_state=random_state, skip_complete=True)
    filled = imputer.fit_transform(mat_df.values)
    out = pd.DataFrame(filled, columns=mat_df.columns, index=mat_df.index)
    n_nan_before = int(np.isnan(mat_df.values).sum())
    n_nan_after  = int(np.isnan(out.values).sum())
    if n_nan_after > 0:
        print(f"  WARNING: {n_nan_after} NaN remain after MICE imputation "
              f"(started with {n_nan_before}) — likely all-NaN column(s).")
    return out


# ── Sample alignment shared by Modules D, C, G ──────────────────────────────
common = sorted(set(spc_df['Sample']) & set(mtb_df['Sample']) & set(meta_df['Sample']))
spc_mat = spc_df.set_index('Sample').loc[common]
mtb_mat = mtb_df.set_index('Sample').loc[common]
meta_mat = meta_df.set_index('Sample').loc[common]
print(f"Common samples (species x metabolites x metadata): {len(common)}")

print("Running MICE imputation on species matrix...")
spc_imp = mice_impute(spc_mat)
print("Running MICE imputation on metabolite matrix...")
mtb_imp = mice_impute(mtb_mat)
print(f"spc_imp: {spc_imp.shape}, mtb_imp: {mtb_imp.shape}")


## Module D — GLASSO Precision Matrix (Partial Correlation Network)

Fits a graphical lasso on the joint species-metabolite matrix to identify **direct**
species-metabolite associations, removing transitive correlations shared through
disease-stage confounding. Unlike the old notebook, edges are not selected by a bare
magnitude threshold alone — a permutation null model (below) gives each candidate edge an
empirical, BH-FDR-corrected p-value.


In [ ]:
# Feature selection for the GLASSO network: with all ~650 features (p > n = 317) the
# CV-selected alpha was so large that no edge survived. Use ~110 features instead:
#   top-60 species by variance + species significant in NB02's partial-correlation pool
#   (cap 80 species total) + all polyamines + top-40 other metabolites by variance.
pc_sig_species = (partial_corr.loc[partial_corr['QValue_partial'] < utils.FDR_THRESHOLD]
                  .sort_values('QValue_partial')['Species'].drop_duplicates().tolist())
sel_spc = list(spc_imp.var().nlargest(60).index)
for s in pc_sig_species:
    if len(sel_spc) >= 80:
        break
    if s in spc_imp.columns and s not in sel_spc:
        sel_spc.append(s)
pa_in_mtb = [c for c in polyamine_columns.values() if c in mtb_imp.columns]
sel_mtb = pa_in_mtb + [c for c in mtb_imp.var().sort_values(ascending=False).index
                       if c not in pa_in_mtb][:40]

spc_d = spc_imp[sel_spc]
mtb_d = mtb_imp[sel_mtb]
joint = np.hstack([
    StandardScaler().fit_transform(spc_d.values),
    StandardScaler().fit_transform(mtb_d.values),
])
joint_cols = list(spc_d.columns) + list(mtb_d.columns)
n_spc = spc_d.shape[1]
n_mtb = mtb_d.shape[1]
print(f"Joint matrix: {joint.shape} ({n_spc} species + {n_mtb} metabolites, "
      f"{len(pa_in_mtb)} polyamines)")

# Solver settings validated on this matrix (statistical review, 2026-10-20): 1,200
# fits with ConvergenceWarning promoted to an error, zero failures. The previous grid
# reached alpha 0.005, where coordinate descent diverges (the overflow / invalid-value
# RuntimeWarnings), and refits at tol=1e-3 with the default inner tolerance
# (enet_tol=1e-4) stalled at a duality gap of ~1e-2. The inner tolerance is what makes
# the solver exact enough. One configuration, GL_KW, is shared by the CV, the observed
# refit and every permutation refit, so both sides of the test are computed identically
# (previously max_iter=300 for the observed fit and 400 for the null).
GL_KW = dict(mode='cd', tol=1e-4, enet_tol=1e-7, max_iter=1000)
ALPHAS = np.logspace(-1.7, -0.5, 13)          # 0.020 .. 0.316
print(f"Fitting GraphicalLassoCV over alphas {np.round(ALPHAS, 4).tolist()} ...")
HAS_GLASSO = True
try:
    glasso = utils.fit_strict(GraphicalLassoCV(alphas=ALPHAS, cv=3, n_jobs=1, **GL_KW), joint)
    _cv_scores = np.asarray(glasso.cv_results_['mean_test_score'], dtype=float)
    if not np.isfinite(_cv_scores).all():
        raise RuntimeError(f"{int((~np.isfinite(_cv_scores)).sum())} alpha(s) scored "
                           "non-finite in cross-validation")
    if not (ALPHAS[0] < glasso.alpha_ < ALPHAS[-1]):
        raise RuntimeError(f"selected alpha {glasso.alpha_:.4f} is at the edge of the grid")
    # Observed network: the same estimator, settings and alpha as every permutation.
    obs_fit = utils.fit_strict(GraphicalLasso(alpha=glasso.alpha_, **GL_KW), joint)
except Exception as fit_exc:
    print(f"!!! Module D SKIPPED -- GLASSO is not numerically sound on this matrix: {fit_exc}")
    HAS_GLASSO = False

if HAS_GLASSO:
    prec = obs_fit.precision_
    print(f"Alpha selected: {glasso.alpha_:.4f}"
          f"{'  (at the edge of the grid)' if glasso.alpha_ in (ALPHAS.min(), ALPHAS.max()) else ''}")
    diag = np.sqrt(np.maximum(np.diag(prec), 1e-10))
    pcorr = -prec / np.outer(diag, diag)
    np.fill_diagonal(pcorr, 0)
    pcorr_cross = pcorr[:n_spc, n_spc:]   # rows=species, cols=metabolites
    print(f"Partial correlation matrix: {pcorr.shape}, cross-block: {pcorr_cross.shape}, "
          f"non-zero cross edges: {int((np.abs(pcorr_cross) > 1e-8).sum())}")


In [ ]:
# ── Permutation null model for GLASSO edges ─────────────────────────────────
# Shuffles the sample correspondence of the metabolite block only (species block stays
# in its original sample order), breaking species-metabolite association while
# preserving each block's own within-block covariance structure, then refits
# GraphicalLasso at the ALREADY-SELECTED alpha (no re-CV — that would be far too slow
# to repeat per permutation). Kept modest (N_PERMUTATIONS) to bound runtime; increase
# for higher p-value resolution / publication use.
N_PERMUTATIONS = 100
MAGNITUDE_PREFILTER = 0.08   # candidate-edge magnitude floor, for reporting only —
                              # final "Significant" status is set by the permutation
                              # test below, not by this threshold alone.

HAS_GLASSO_NULL = False
glasso_edges = pd.DataFrame(columns=['Species', 'Metabolite', 'PartialCorr', 'AbsPartialCorr',
                                      'Empirical_PValue', 'QValue', 'Significant', 'Validated'])

if HAS_GLASSO:
    rng = np.random.default_rng(42)
    fixed_alpha = glasso.alpha_
    spc_scaled = StandardScaler().fit_transform(spc_d.values)
    mtb_scaled = StandardScaler().fit_transform(mtb_d.values)

    null_abs_list, n_perm_failed = [], 0
    for perm_i in range(N_PERMUTATIONS):
        perm_idx = rng.permutation(len(common))
        joint_perm = np.hstack([spc_scaled, mtb_scaled[perm_idx]])
        try:
            gl_perm = utils.fit_strict(GraphicalLasso(alpha=fixed_alpha, **GL_KW), joint_perm)
        except utils.NUMERICAL_FAILURES:
            # Counted as an exceedance for EVERY edge, never dropped. Dropping failed fits
            # shrinks the null silently and can only make p-values smaller.
            n_perm_failed += 1
            null_abs_list.append(np.full((n_spc, n_mtb), np.inf))
            continue
        prec_perm = gl_perm.precision_
        diag_perm = np.sqrt(np.maximum(np.diag(prec_perm), 1e-10))
        pcorr_perm = -prec_perm / np.outer(diag_perm, diag_perm)
        null_abs_list.append(np.abs(pcorr_perm[:n_spc, n_spc:]))

    n_valid_perms = len(null_abs_list)          # == N_PERMUTATIONS: failures are kept
    print(f"Permutations: {N_PERMUTATIONS} run, {N_PERMUTATIONS - n_perm_failed} converged, "
          f"{n_perm_failed} failed (counted as exceedances).")
    if n_perm_failed:
        print(f"!!! {n_perm_failed} GLASSO permutation fit(s) failed to converge.")

    observed_abs = np.abs(pcorr_cross)
    rows, cols = np.where(observed_abs >= MAGNITUDE_PREFILTER)
    print(f"Candidate edges (|partial_corr| >= {MAGNITUDE_PREFILTER}): {len(rows)}")

    if n_perm_failed > 0.05 * N_PERMUTATIONS:
        print(f"!!! {n_perm_failed}/{N_PERMUTATIONS} permutation fits failed (>5%) -- "
              f"permutation-based significance is unreliable. Saving magnitude-threshold "
              f"candidate edges with Validated=False (UNVALIDATED -- no null-model support) "
              f"rather than claiming significance.")
        glasso_edges = pd.DataFrame({
            'Species':          [spc_d.columns[r] for r in rows],
            'Metabolite':       [mtb_d.columns[c] for c in cols],
            'PartialCorr':      [pcorr_cross[r, c] for r, c in zip(rows, cols)],
        })
        glasso_edges['AbsPartialCorr']   = glasso_edges['PartialCorr'].abs()
        glasso_edges['Empirical_PValue'] = np.nan
        glasso_edges['QValue']           = np.nan
        glasso_edges['Significant']      = False
        glasso_edges['Validated']        = False
    else:
        null_stack = np.stack(null_abs_list, axis=0)   # (n_valid_perms, n_spc, n_mtb)
        counts_ge = (null_stack >= observed_abs[None, :, :]).sum(axis=0)
        emp_p = (counts_ge + 1) / (n_valid_perms + 1)

        all_rows, all_cols = np.where(np.ones_like(observed_abs, dtype=bool))
        full_df = pd.DataFrame({
            'Species':          [spc_d.columns[r] for r in all_rows],
            'Metabolite':       [mtb_d.columns[c] for c in all_cols],
            'PartialCorr':      [pcorr_cross[r, c]  for r, c in zip(all_rows, all_cols)],
            'Empirical_PValue': [emp_p[r, c]        for r, c in zip(all_rows, all_cols)],
        })
        full_df['AbsPartialCorr'] = full_df['PartialCorr'].abs()
        _, qvals, _, _ = multipletests(full_df['Empirical_PValue'], method='fdr_bh')
        full_df['QValue'] = qvals
        # Westfall-Young max-statistic p-value (FWER over ALL cross edges): per-edge BH
        # cannot work here -- with ~n_spc*n_mtb edges and N_PERMUTATIONS permutations the
        # smallest attainable BH q is far above 0.05. The max-T null compares each
        # observed |partial corr| with the largest |partial corr| of every permuted fit.
        max_null = null_stack.reshape(n_valid_perms, -1).max(axis=1)
        p_maxT = (np.sum(max_null[:, None] >= observed_abs.ravel()[None, :], axis=0) + 1) / (n_valid_perms + 1)
        p_maxT = p_maxT.reshape(observed_abs.shape)
        full_df['P_maxT_FWER'] = [p_maxT[r, c] for r, c in zip(all_rows, all_cols)]
        full_df['Significant'] = full_df['P_maxT_FWER'] < 0.05
        print(f"max-T null: 95th percentile of max |partial corr| = {np.quantile(max_null, 0.95):.3f}")
        full_df['Validated'] = True
        HAS_GLASSO_NULL = True

        # Keep magnitude-prefiltered-OR-significant edges (readable table; every row
        # still carries its real permutation-based QValue/Significant status).
        glasso_edges = full_df[
            (full_df['AbsPartialCorr'] >= MAGNITUDE_PREFILTER) | full_df['Significant']
        ].sort_values('AbsPartialCorr', ascending=False).reset_index(drop=True)

    # Numerical diagnostics travel with the edges, so a reader of the table alone can see
    # how the null was obtained.
    glasso_edges['Alpha'] = fixed_alpha
    glasso_edges['Alpha_at_grid_edge'] = bool(fixed_alpha in (ALPHAS.min(), ALPHAS.max()))
    glasso_edges['N_permutations'] = N_PERMUTATIONS
    glasso_edges['N_perm_failed'] = n_perm_failed
    glasso_edges.to_csv(TABLES / 'd_glasso_partial_correlation_edges_v2.csv', index=False)
    n_sig = int(glasso_edges['Significant'].sum()) if 'Significant' in glasso_edges else 0
    print(f"GLASSO edges saved: {len(glasso_edges)} (max-T permutation FWER-significant: {n_sig})")
    print("Saved: d_glasso_partial_correlation_edges_v2.csv")
else:
    print("Module D skipped (GraphicalLassoCV did not converge) — no edges to permute.")
    glasso_edges.to_csv(TABLES / 'd_glasso_partial_correlation_edges_v2.csv', index=False)
    print("Saved empty d_glasso_partial_correlation_edges_v2.csv (Module D unavailable).")


In [ ]:
PA_KEGG_IDS = set(utils.POLYAMINE_KEGG.keys())
PA_COLS = {c for c in mtb_imp.columns
           if any(c.startswith(k + '_') for k in PA_KEGG_IDS) or c in polyamine_columns.values()}

if HAS_GLASSO and not glasso_edges.empty:
    import networkx as nx

    top_edges = glasso_edges.head(60)
    G = nx.Graph()
    for _, row in top_edges.iterrows():
        G.add_node(row['Species'], node_type='species')
        G.add_node(row['Metabolite'], node_type='metabolite')
        G.add_edge(row['Species'], row['Metabolite'],
                   weight=abs(row['PartialCorr']),
                   sign='pos' if row['PartialCorr'] > 0 else 'neg')

    color_map, size_map = {}, {}
    for n in G.nodes():
        if G.nodes[n].get('node_type') == 'species':
            color_map[n] = '#3498db'; size_map[n] = 160
        elif n in PA_COLS:
            color_map[n] = '#e74c3c'; size_map[n] = 400
        else:
            color_map[n] = '#2ecc71'; size_map[n] = 200
    node_colors = [color_map[n] for n in G.nodes()]
    node_sizes  = [size_map[n] for n in G.nodes()]
    edge_colors = ['#e74c3c' if G[u][v].get('sign') == 'neg' else '#27ae60'
                   for u, v in G.edges()]

    fig, ax = plt.subplots(figsize=(18, 12))
    pos = nx.spring_layout(G, k=1.8, seed=42)
    nx.draw(G, pos, node_color=node_colors, node_size=node_sizes,
            edge_color=edge_colors, alpha=0.75, ax=ax, with_labels=False, width=0.9)
    labels = {n: (n.split('_', 1)[-1] if '_' in n else n)[:28]
              for n in G.nodes()
              if G.nodes[n].get('node_type') == 'metabolite' or G.degree(n) >= 4}
    nx.draw_networkx_labels(G, pos, labels, font_size=7, ax=ax)

    legend = [
        mpatches.Patch(color='#e74c3c', label='Polyamine metabolite'),
        mpatches.Patch(color='#2ecc71', label='Other metabolite'),
        mpatches.Patch(color='#3498db', label='Species'),
    ]
    ax.legend(handles=legend, fontsize=9)
    validated_note = 'permutation-FDR validated' if HAS_GLASSO_NULL else 'UNVALIDATED (magnitude only)'
    ax.set_title(f'GLASSO Partial Correlation Network (top {len(top_edges)} edges, {validated_note})\n'
                 'Red edges = negative partial correlation, green = positive', fontsize=12)
    plt.tight_layout()
    out_d = FIGURES / 'd_glasso_network_v2.png'
    fig.savefig(out_d, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {out_d}")
else:
    print("Module D network figure skipped — no GLASSO edges available.")


## Module B — AGORA2 Species-Level Enzyme-Profile Evidence

Replaces the old Module B (KEGG REST genus survey / NCBI Entrez assembly lookup / UniProt
keyword search — sub-modules B1/B2/B3, all dropped) with the same AGORA2 species-level
lookup used in NB08: `utils.build_agora2_ec_index()` + `utils.species_encodes_enzyme()`.
The old genus-org-code approach (`genus[:4] in org.lower()`) inferred a *study* species'
enzyme content from *any* KEGG organism sharing a 4-character genus prefix — essentially
never biologically meaningful. Here we ask the direct question against the species' own
AGORA2 genome-scale reconstruction, for **every** polyamine KEGG compound (not just the one
each species was ranked as a top producer of), giving a fuller per-species enzyme profile
than NB08's per-target-polyamine `r1_agora2_producer_enrichment_v2.csv` — which we also
load and cross-check against below (closing the "must genuinely consume NB08's tables"
requirement).

Guarded on `HAS_PA_PRODUCERS`: if NB03's producer list is empty, this entire module is
skipped with a clear message, not silently indexed into and crashed.


In [ ]:
HAS_MODULE_B = False
b_profile_df = pd.DataFrame(columns=['Species', 'Genus'] + list(utils.POLYAMINE_KEGG.keys())
                             + ['Total_AGORA2_Encoded', 'AGORA2_Model_Available'])

if not HAS_PA_PRODUCERS or not utils.AGORA2_ENABLED:
    print("SKIPPED: Module B needs NB03's pa_producers AND a genomic reference "
          "index; the index is unavailable (AGORA2 disabled). No profile table is built.")
else:
    agora2_ec_index = None  # unreachable while AGORA2 is disabled (guard above)
    print("Genomic reference index: unavailable")


    def genus_of(species_col):
        g = species_col.split()[0] if species_col.split() else species_col
        for suf in ('_A', '_B', '_C', '_D', '_E', '_F'):
            if g.endswith(suf):
                g = g[:-len(suf)]
        return g

    producer_species = sorted(pa_producers['Species'].unique())
    print(f"Building AGORA2 enzyme profile for {len(producer_species)} producer species "
          f"across {len(utils.POLYAMINE_KEGG)} polyamine KEGG compounds...")

    rows_b = []
    for sp in producer_species:
        hits = {}
        any_model = False
        for kegg_id in utils.POLYAMINE_KEGG:
            result = None  # unreachable while AGORA2 is disabled
            if result is not None:
                any_model = True
            hits[kegg_id] = bool(result) if result is not None else False
        rows_b.append({
            'Species': sp,
            'Genus':   genus_of(sp),
            **hits,
            'Total_AGORA2_Encoded':   sum(hits.values()),
            'AGORA2_Model_Available': any_model,
        })

    b_profile_df = pd.DataFrame(rows_b)
    b_profile_df.to_csv(TABLES / 'b_agora2_producer_enzyme_profile_v2.csv', index=False)
    HAS_MODULE_B = True

    n_with_model = int(b_profile_df['AGORA2_Model_Available'].sum())
    print(f"\nAGORA2 enzyme profile built for {len(b_profile_df)} producer species "
          f"({n_with_model} with an AGORA2 reconstruction available).")
    print(b_profile_df['Total_AGORA2_Encoded'].value_counts().sort_index().to_string())
    print("Saved: b_agora2_producer_enzyme_profile_v2.csv")

    # ── Cross-check against NB08's per-target r1 table (genuinely consuming NB08) ──
    if HAS_R1:
        n_r1_confirmed = int((r1_df['Encodes_Enzyme'] == True).sum())
        n_r1_refuted   = int((r1_df['Encodes_Enzyme'] == False).sum())
        n_r1_no_model  = int(r1_df['Encodes_Enzyme'].isna().sum())
        print(f"\nCross-check vs NB08 r1_agora2_producer_enrichment_v2.csv "
              f"({len(r1_df)} per-target-polyamine rows):")
        print(f"  AGORA2-confirmed producers: {n_r1_confirmed}")
        print(f"  AGORA2-refuted producers:   {n_r1_refuted}")
        print(f"  No AGORA2 model available:  {n_r1_no_model}")
    else:
        print("\n(NB08's r1 table unavailable — cross-check skipped; see WARNING above.)")


## Module C — Curated Pathway Mapping + Species-Pathway Correlations

**Honesty fix:** the old notebook defined a `reactome_query()` function whose name and
docstring implied a live Reactome REST content-service call, but the pathway assignments
actually *used* downstream came from a small hardcoded keyword-substring dictionary that
`reactome_query()` was never wired into. There is no live pathway API call in this
notebook. `CURATED_PATHWAY_MAP` below is a manually-curated reference (the R-HSA
identifiers are real Reactome pathway IDs, looked up by hand from the Reactome pathway
browser) — labeled as exactly that, not dressed up as an automated query result. The dict
is defined in full before any code that consults it (fixing the forward-reference pattern
in the same spot the old notebook had it).

Species-pathway correlations use the same MICE-imputed `spc_imp` / `mtb_imp` matrices as
Module D (one missing-data policy, not a third one).


In [ ]:
import re

# Curated reference (R-HSA identifiers looked up by hand -- NOT a live API), keyed by KEGG
# compound ID so that e.g. hippurate is never matched as "urate" or hydroxyurea as "urea".
CURATED_PATHWAY_BY_KEGG = {
    'C00191': ('glucuronate',        {'R-HSA-5661977': 'Steroid hormone biosynthesis (glucuronate conjugation)'}),
    'C00366': ('urate',              {'R-HSA-74217':   'Purine catabolism'}),
    'C00086': ('urea',               {'R-HSA-70635':   'Urea cycle'}),
    'C00491': ('cystine',            {'R-HSA-1614517': 'Sulfur amino acid metabolism'}),
    'C06104': ('adipate',            {'R-HSA-77289':   'Mitochondrial fatty acid beta-oxidation (VLCFA)'}),
    'C08277': ('sebacate',           {'R-HSA-77289':   'Mitochondrial fatty acid beta-oxidation (VLCFA)'}),
    'C06103': ('6-hydroxyhexanoate', {'R-HSA-204005':  'Omega-oxidation of fatty acids'}),
    'C02656': ('pimelate',           {'R-HSA-77289':   'Mitochondrial fatty acid beta-oxidation (VLCFA)'}),
    'C00134': ('putrescine',         {'R-HSA-1430728': 'Polyamine biosynthesis'}),
    'C00315': ('spermidine',         {'R-HSA-1430728': 'Polyamine biosynthesis'}),
    'C00750': ('spermine',           {'R-HSA-1430728': 'Polyamine biosynthesis'}),
    'C00077': ('ornithine',          {'R-HSA-70268':   'Urea cycle / ornithine metabolism'}),
}

def kegg_of(col):
    kid = (mtb_kegg_map or {}).get(col)
    if kid:
        return kid
    m = re.match(r'(C\d{5})_', col)
    return m.group(1) if m else None

def clean_mtb_name(col):
    return col.split('_', 1)[-1].lower() if '_' in col else col.lower()

def whole_word(keyword, name):
    # keyword must not be glued to letters/digits on either side ("urate" != "hippurate")
    return re.search(r'(?<![a-z0-9])' + re.escape(keyword) + r'(?![a-z0-9])', name) is not None

mapping_rows = []
for col in mtb_imp.columns:
    kid = kegg_of(col)
    matched, how = {}, None
    if kid in CURATED_PATHWAY_BY_KEGG:
        matched, how = CURATED_PATHWAY_BY_KEGG[kid][1], 'KEGG_ID'
    else:
        name = clean_mtb_name(col)
        for keyword, paths in CURATED_PATHWAY_BY_KEGG.values():
            if whole_word(keyword, name):
                matched.update(paths); how = 'whole-word name'
    for path_id, path_name in matched.items():
        mapping_rows.append({'Feature': col, 'KEGG_ID': kid, 'Matched_by': how,
                             'Reactome_ID': path_id, 'Reactome_Pathway': path_name})

pathway_mapping_df = pd.DataFrame(mapping_rows, columns=['Feature', 'KEGG_ID', 'Matched_by',
                                                         'Reactome_ID', 'Reactome_Pathway'])
pathway_mapping_df.to_csv(TABLES / 'c_pathway_mapping_v2.csv', index=False)
print(f"Curated pathway mapping (metabolite -> Reactome pathway): {len(pathway_mapping_df)} matched features")
print(pathway_mapping_df.to_string(index=False))
print("Saved: c_pathway_mapping_v2.csv  (curated reference — NOT a live Reactome API result)")


In [ ]:
# Pathway score = mean of MICE-imputed abundances of every metabolite matched to that
# pathway (same imputed matrix as Module D — one missing-data policy).
pathway_scores = {}
for path_name in pathway_mapping_df['Reactome_Pathway'].unique():
    cols = pathway_mapping_df.loc[pathway_mapping_df['Reactome_Pathway'] == path_name, 'Feature'].unique()
    cols = [c for c in cols if c in mtb_imp.columns]
    if cols:
        pathway_scores[path_name] = mtb_imp[cols].mean(axis=1).values

print("Pathway scores defined:", list(pathway_scores.keys()))

corr_rows = []
for pw_name, pw_vals in pathway_scores.items():
    for sp in spc_imp.columns:
        x = spc_imp[sp].values
        if np.std(x) == 0:
            continue
        rho, p = spearmanr(x, pw_vals)
        if np.isnan(rho):
            continue
        corr_rows.append({'Pathway': pw_name, 'Species': sp, 'Rho': rho, 'PValue': p})

pw_corr_df = pd.DataFrame(corr_rows)
if not pw_corr_df.empty:
    _, pw_corr_df['QValue'], _, _ = multipletests(pw_corr_df['PValue'], method='fdr_bh')
    sig = pw_corr_df[pw_corr_df['QValue'] < utils.FDR_THRESHOLD].sort_values('Rho', ascending=False)
else:
    sig = pd.DataFrame()
pw_corr_df.to_csv(TABLES / 'c_species_pathway_correlations_v2.csv', index=False)
print(f"\nSignificant species-pathway correlations (q<{utils.FDR_THRESHOLD}): {len(sig)}")
if not sig.empty:
    print(sig.groupby('Pathway').size().to_string())
print("Saved: c_species_pathway_correlations_v2.csv")

# ── Cross-reference against NB08's r8 polyamine/dicarboxylic-acid overlap ──────
if HAS_R8 and not r8_df.empty and not sig.empty:
    r8_species = set(r8_df['Species'].unique())
    module_c_species = set(sig['Species'].unique())
    overlap = r8_species & module_c_species
    print(f"\nCross-reference vs NB08 r8_polyamine_dicarb_overlap_v2.csv: "
          f"{len(overlap)}/{len(module_c_species)} Module-C-significant species also appear "
          f"in NB08's polyamine/dicarboxylic-acid overlap table.")
elif not HAS_R8:
    print("\n(NB08's r8 table unavailable — cross-reference skipped; see WARNING above.)")
else:
    print("\n(No significant Module C species or empty r8 table — cross-reference skipped.)")


In [ ]:
if not sig.empty:
    pivot = sig.pivot_table(index='Species', columns='Pathway', values='Rho', aggfunc='first')
    top_sp = pivot.abs().max(axis=1).nlargest(25).index
    pivot_top = pivot.loc[pivot.index.isin(top_sp)].fillna(0)

    fig, ax = plt.subplots(figsize=(8, max(6, len(pivot_top) * 0.35)))
    sns.heatmap(pivot_top, cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
                ax=ax, linewidths=0.3, cbar_kws={'label': 'Spearman Rho'})
    ax.set_title('Species x Curated-Pathway Scores\n(top 25 species by max |Rho|, q<0.05)', fontsize=11)
    ax.set_xlabel('')
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)
    plt.tight_layout()
    out_c = FIGURES / 'c_species_pathway_heatmap_v2.png'
    fig.savefig(out_c, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {out_c}")
else:
    print("Module C figure skipped — no significant species-pathway correlations.")


## Module G — AGORA2-defined functional guild scores x MOFA+ factors

Per-sample guild scores = mean CLR abundance (MICE-imputed `spc_imp`) of the reduced species
whose **AGORA2 model encodes the guild's enzymes** (`utils.POLYAMINE_EC` for the polyamines
proper, `utils.SCFA_EC`, `utils.PURINE_CATAB_EC`). The previous version used hand-weighted
genus lists, the polyamine one copied from NB03's SHAP top genera -- which made any
"polyamine guild tracks stage" result circular. Species without an AGORA2 model cannot be
assigned and are left out (member counts are printed). Scores are correlated with **every**
MOFA+ factor; the stage figure uses `utils.STAGE_GROUP_ORDER`.

In [ ]:
# Genomic guilds need a genomic reference index, unavailable in this run
# (utils.AGORA2_ENABLED is False). Every guild is therefore empty and its score
# NaN, which downstream must read as 'not computable', never as 'no members'.
agora2_ec_index_g = None

GUILD_ECS = {
    'Polyamine_biosyn':  sorted({ec for k in utils.POLYAMINE_PATHWAY_KEGG for ec in utils.POLYAMINE_EC[k]}),
    'SCFA_butyrate':     utils.SCFA_EC['butyrate'],
    'SCFA_propionate':   utils.SCFA_EC['propionate'],
    'Purine_catabolism': utils.PURINE_CATAB_EC,
}

enrichment_scores, guild_members = {}, {}
n_with_model = 0  # no genomic reference index

print(f"Reduced species with an AGORA2 model: {n_with_model}/{spc_imp.shape[1]}")
for pw_name, ecs in GUILD_ECS.items():
    members = [sp for sp in spc_imp.columns
               if False]  # no genomic reference index: membership not callable
    guild_members[pw_name] = members
    enrichment_scores[pw_name] = (spc_imp[members].mean(axis=1).values if members
                                  else np.full(len(common), np.nan))
    print(f"  {pw_name:20s}: ECs {ecs} -> {len(members)} member species")

pd.DataFrame([(g, s) for g, ms in guild_members.items() for s in ms],
             columns=['Guild', 'Species']).to_csv(TABLES / 'g_guild_members_v2.csv', index=False)

pw_cols = [g for g in GUILD_ECS if guild_members[g]]
enrich_df = pd.DataFrame({g: enrichment_scores[g] for g in pw_cols}, index=common)
enrich_df['Sample'] = common
enrich_df = enrich_df.merge(
    meta_df[['Sample', 'Study.Group', 'CRC.Stage.Numeric', 'Stage.Group']], on='Sample')
print(f"\nGuild score matrix: {enrich_df.shape}")
enrich_df.to_csv(TABLES / 'g_pathway_enrichment_scores_v2.csv', index=False)
print("Saved: g_pathway_enrichment_scores_v2.csv, g_guild_members_v2.csv")


In [ ]:
pw_factor_df = pd.DataFrame()

if HAS_MOFA_SCORES and not mofa_scores.empty:
    factor_cols = [c for c in mofa_scores.columns if c.startswith('Factor')]
    mofa_common = sorted(set(common) & set(mofa_scores['Sample']))
    if len(mofa_common) / max(len(common), 1) < 0.95:
        print(f"WARNING: only {len(mofa_common)}/{len(common)} samples have MOFA+ factor "
              f"scores — correlations below use this reduced, structurally-present-only "
              f"subset (not imputed; a sample missing from the MOFA+ fit is not a NaN "
              f"cell to fill in).")

    mofa_aligned  = mofa_scores.set_index('Sample').loc[mofa_common]
    enrich_aligned = enrich_df.set_index('Sample').loc[mofa_common]

    corr_rows = []
    for pw in pw_cols:
        for fc in factor_cols:
            rho, p = spearmanr(enrich_aligned[pw].values, mofa_aligned[fc].values)
            if not np.isnan(rho):
                corr_rows.append({'Pathway': pw, 'Factor': fc, 'Rho': rho, 'PValue': p})

    pw_factor_long = pd.DataFrame(corr_rows)
    if not pw_factor_long.empty:
        _, pw_factor_long['QValue'], _, _ = multipletests(pw_factor_long['PValue'], method='fdr_bh')
        pw_factor_long['Significant'] = pw_factor_long['QValue'] < utils.FDR_THRESHOLD
    pw_factor_long.to_csv(TABLES / 'g_pathway_factor_correlations_v2.csv', index=False)
    print(f"Pathway x MOFA+ factor correlations (BH-FDR corrected, {len(factor_cols)} factors): "
          f"{len(pw_factor_long)} tests, "
          f"{int(pw_factor_long['Significant'].sum()) if not pw_factor_long.empty else 0} significant")
    print("Saved: g_pathway_factor_correlations_v2.csv")

    if not pw_factor_long.empty:
        pw_factor_df = pw_factor_long.pivot_table(index='Pathway', columns='Factor', values='Rho')
        pval_df = pw_factor_long.pivot_table(index='Pathway', columns='Factor', values='PValue')

        fig, ax = plt.subplots(figsize=(max(10, 1.0 * len(pw_factor_df.columns)), 5))
        sns.heatmap(pw_factor_df, cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5, ax=ax,
                    annot=True, fmt='.2f', annot_kws={'size': 7}, linewidths=0.4,
                    cbar_kws={'label': 'Spearman Rho'})
        for i, pw in enumerate(pw_factor_df.index):
            for j, fc in enumerate(pw_factor_df.columns):
                p = pval_df.loc[pw, fc] if fc in pval_df.columns else np.nan
                star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
                if star:
                    ax.text(j + 0.5, i + 0.85, star, ha='center', va='center', fontsize=8, color='black')
        ax.set_title('AGORA2-defined guild scores x MOFA+ Factors (all fitted factors)\n'
                      '(*p<0.05, **p<0.01, ***p<0.001, nominal — see QValue column for BH-FDR)', fontsize=10)
        ax.set_xlabel('MOFA+ Factor'); ax.set_ylabel('Microbial Pathway')
        plt.tight_layout()
        out_g1 = FIGURES / 'g_pathway_mofa_factor_heatmap_v2.png'
        fig.savefig(out_g1, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved: {out_g1}")
else:
    print("SKIPPED: Module G's MOFA-factor correlation step requires mofa_factor_scores_v2.csv, "
          "which is unavailable. g_pathway_factor_correlations_v2.csv is written empty.")
    pd.DataFrame(columns=['Pathway', 'Factor', 'Rho', 'PValue', 'QValue', 'Significant']).to_csv(
        TABLES / 'g_pathway_factor_correlations_v2.csv', index=False)


In [ ]:
# Stage-stratified boxplots — utils.STAGE_GROUP_MAP / STAGE_GROUP_ORDER ONLY.
# No local stage-ordering dict (the old notebook's `stage_order`/`stage_pal` here
# hand-duplicated the pre-fix fine-grained ordering with HS/MP in the wrong place).
if not pw_cols:
    # Every guild is empty while no genomic reference index is built (AGORA2 is
    # disabled; a gutSMASH index is pending), so there is nothing to stratify.
    # Skipping is the honest output: an empty figure would read as a result.
    print('SKIPPED: stage-stratified guild boxplots -- no guild is computable '
          '(no genomic reference index, so every guild has zero members).')
else:
    stage_present = [s for s in utils.STAGE_GROUP_ORDER if s in enrich_df['Stage.Group'].values]
    stage_pal = dict(zip(utils.STAGE_GROUP_ORDER,
                          ['#2ecc71', '#f1c40f', '#e67e22', '#8e44ad']))

    fig, axes = plt.subplots(1, len(pw_cols), figsize=(4.5 * len(pw_cols), 5))
    if len(pw_cols) == 1:
        axes = [axes]

    for ax, pw in zip(axes, pw_cols):
        data = [enrich_df.loc[enrich_df['Stage.Group'] == sg, pw].dropna().values for sg in stage_present]
        bp = ax.boxplot(data, patch_artist=True, medianprops=dict(color='black', linewidth=1.5))
        for patch, sg in zip(bp['boxes'], stage_present):
            patch.set_facecolor(stage_pal.get(sg, '#cccccc'))
            patch.set_alpha(0.75)
        groups_nonzero = [d for d in data if len(d) > 1]
        if len(groups_nonzero) >= 2:
            stat, kw_p = kruskal(*groups_nonzero)
            ax.set_title(f'{pw}\nKW p={kw_p:.3f}', fontsize=9)
        else:
            ax.set_title(f'{pw}\nKW n.a. (<2 groups with data)', fontsize=9)
        ax.set_xticks(range(1, len(stage_present) + 1))
        ax.set_xticklabels(stage_present, rotation=30, ha='right', fontsize=8)
        ax.grid(True, axis='y', alpha=0.3)
        ax.axhline(0, color='grey', ls='--', lw=0.7)

    # The previous title listed "Healthy / Adenoma / Early_CRC / Advanced_CRC" as
    # STAGE_GROUP_ORDER. Adenoma (MP) is excluded from the study; the order is three
    # groups, with PostSurgery off the ordinal axis.
    plt.suptitle('Stage-Stratified Genomic Guild Scores\n'
                  '(Healthy / Early_CRC / Advanced_CRC — utils.STAGE_GROUP_ORDER)',
                  fontsize=12, fontweight='bold')
    plt.tight_layout()
    out_g2 = FIGURES / 'g_pathway_stage_boxplots_v2.png'
    fig.savefig(out_g2, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {out_g2}")


## Final Synthesis — Cross-Notebook Evidence Triangulation

**Candidate pool (independent of SHAP):** every species-polyamine pair that is significant
and **positive** in NB02's covariate-adjusted partial Spearman correlation
(`QValue_partial < 0.05`, `Rho_partial > 0`). The previous version took its candidates from
NB03's SHAP producer list, so the SHAP source could only ever say "yes".

Each candidate is then scored against three sources:

- **Source A — NB03 SHAP**: available if NB03 modelled that polyamine; supports the pair if
  the species is among that polyamine's top-20 OOF-SHAP species with a **positive** direction,
  a stable bootstrap rank, **and** the model itself beat its permutation null
  (`Model_perm_q < 0.05`).
- **Source B — AGORA2 enzyme**: available if the species has an AGORA2 model; supports if the
  model encodes an enzyme producing that polyamine (`utils.species_encodes_enzyme`).
- **Source C — MOFA co-loading**: available if both features are in the MOFA+ fit; supports
  if they load on a shared factor (|weight| > 0.10) **with the same sign** (positive
  co-variation).

Missing coverage is reported as `..._Available = False` and `..._Support = None`, never as
evidence against.

In [ ]:
kegg_by_name = {v: k for k, v in utils.POLYAMINE_KEGG.items()}


def resolve_polyamine_column(kegg_id):
    if kegg_id is None:
        return None
    if kegg_id in polyamine_columns and polyamine_columns[kegg_id] in mtb_df.columns:
        return polyamine_columns[kegg_id]
    inv_map = {v: k for k, v in (mtb_kegg_map or {}).items()}
    if kegg_id in inv_map and inv_map[kegg_id] in mtb_df.columns:
        return inv_map[kegg_id]
    for col in mtb_df.columns:
        if col.startswith(kegg_id + '_'):
            return col
    return None


# ── Source A: NB03 SHAP -- which polyamines were modelled, and which pairs pass ──
modelled_polyamines = set(pa_producers_all['Polyamine']) if not pa_producers_all.empty else set()
shap_support = set()
if HAS_PA_PRODUCERS:
    _ok = pa_producers[pa_producers['Rank_stable'].astype(bool)]
    if 'Model_perm_q' in _ok.columns:
        _ok = _ok[_ok['Model_perm_q'] < utils.FDR_THRESHOLD]
    shap_support = set(zip(_ok['Species'], _ok['Polyamine']))
print(f"Source A: {len(modelled_polyamines)} polyamines modelled in NB03; "
      f"{len(shap_support)} (species, polyamine) pairs pass the SHAP criteria")

# ── Source B: AGORA2 enzyme evidence, computed directly for every candidate ──
agora2_ec_index_s = None  # genomic reference index unavailable: Source B returns None

# ── Source C lookups: NB06 MOFA factor-loading evidence (all factors) ──────
MOFA_W_THRESH = 0.10
species_features_all, mtb_features_all = set(), set()
species_strong_factors, mtb_strong_factors = {}, {}

if HAS_MOFA_LOADINGS and not mofa_loadings.empty:
    cols_lower = {c.lower(): c for c in mofa_loadings.columns}
    view_col = cols_lower.get('view', 'View')
    feat_col = cols_lower.get('feature', 'Feature')
    fac_col  = cols_lower.get('factor', 'Factor')
    wt_col   = cols_lower.get('weight', 'Weight')

    species_features_all = set(mofa_loadings.loc[mofa_loadings[view_col] == 'species', feat_col])
    mtb_features_all     = set(mofa_loadings.loc[mofa_loadings[view_col] == 'metabolites', feat_col])

    strong = mofa_loadings[mofa_loadings[wt_col].abs() > MOFA_W_THRESH]
    # {feature: {factor: sign}} for strong loadings, so co-loading can require the same sign
    species_strong_factors = {
        f: dict(zip(g[fac_col], np.sign(g[wt_col])))
        for f, g in strong[strong[view_col] == 'species'].groupby(feat_col)
    }
    mtb_strong_factors = {
        f: dict(zip(g[fac_col], np.sign(g[wt_col])))
        for f, g in strong[strong[view_col] == 'metabolites'].groupby(feat_col)
    }

# ── Candidate pool: NB02 partial-correlation-significant, positive pairs ───
cand_df = partial_corr[(partial_corr['QValue_partial'] < utils.FDR_THRESHOLD)
                       & (partial_corr['Rho_partial'] > 0)
                       & partial_corr['Polyamine'].notna()]
pc_info = cand_df.set_index(['Species', 'Polyamine'])[['Rho_partial', 'QValue_partial']].to_dict('index')
candidate_pairs = sorted(pc_info.keys())

print(f"Candidate species-polyamine pairs (NB02 partial corr q<{utils.FDR_THRESHOLD}, rho>0): "
      f"{len(candidate_pairs)}")
if not candidate_pairs:
    print("!!! No candidate pairs -- final_evidence_synthesis_v2.csv will be written empty "
          "(with the correct schema), not silently omitted.")


In [ ]:
SYNTH_COLUMNS = [
    'Species', 'Polyamine', 'KEGG_ID', 'Rho_partial', 'QValue_partial',
    'Source_A_ShapStable_Available', 'Source_A_ShapStable_Support',
    'Source_B_AGORA2_Available', 'Source_B_AGORA2_Support',
    'Source_C_MOFA_Available', 'Source_C_MOFA_Support',
    'N_Sources_Available', 'N_Sources_Supporting', 'Evidence_Level',
]


def evidence_level(n_support):
    if n_support >= 3:
        return 'Strong (3/3 supporting)'
    if n_support == 2:
        return 'Moderate (2/3 supporting)'
    if n_support == 1:
        return 'Weak (1/3 supporting)'
    return 'None (0/available supporting)'


rows_synth = []
for sp, pa in candidate_pairs:
    kegg_id = kegg_by_name.get(pa)
    mtb_col = resolve_polyamine_column(kegg_id) if kegg_id else None

    a_avail = pa in modelled_polyamines
    a_support = ((sp, pa) in shap_support) if a_avail else None

    b_res = None  # Source B (genomic) unavailable: no reference index
    b_avail = b_res is not None
    b_support = bool(b_res) if b_avail else None

    c_avail = bool(HAS_MOFA_LOADINGS and sp in species_features_all
                   and mtb_col is not None and mtb_col in mtb_features_all)
    if c_avail:
        sf, mf = species_strong_factors.get(sp, {}), mtb_strong_factors.get(mtb_col, {})
        c_support = any(sf[f] == mf[f] for f in set(sf) & set(mf))
    else:
        c_support = None

    n_avail = sum([a_avail, b_avail, c_avail])
    n_support = sum(1 for s in (a_support, b_support, c_support) if s is True)

    rows_synth.append({
        'Species': sp, 'Polyamine': pa, 'KEGG_ID': kegg_id,
        'Rho_partial': pc_info[(sp, pa)]['Rho_partial'],
        'QValue_partial': pc_info[(sp, pa)]['QValue_partial'],
        'Source_A_ShapStable_Available': a_avail, 'Source_A_ShapStable_Support': a_support,
        'Source_B_AGORA2_Available':     b_avail, 'Source_B_AGORA2_Support':     b_support,
        'Source_C_MOFA_Available':       c_avail, 'Source_C_MOFA_Support':       c_support,
        'N_Sources_Available':   n_avail,
        'N_Sources_Supporting':  n_support,
        'Evidence_Level':        evidence_level(n_support),
    })

synth_df = pd.DataFrame(rows_synth, columns=SYNTH_COLUMNS)
synth_df = synth_df.sort_values(['N_Sources_Supporting', 'N_Sources_Available', 'Rho_partial'],
                                ascending=False).reset_index(drop=True)
synth_df.to_csv(TABLES / 'final_evidence_synthesis_v2.csv', index=False)
print(f"Saved: final_evidence_synthesis_v2.csv  ({len(synth_df)} pairs)")

if not synth_df.empty:
    print("\n--- Source availability coverage ---")
    avail_counts = synth_df['N_Sources_Available'].value_counts()
    for k in [3, 2, 1, 0]:
        print(f"  {k} source(s) available: {int(avail_counts.get(k, 0))} pairs")

    print("\n--- Evidence-score distribution (supporting sources per pair) ---")
    support_counts = synth_df['N_Sources_Supporting'].value_counts()
    for k in [3, 2, 1, 0]:
        print(f"  {k} source(s) supporting: {int(support_counts.get(k, 0))} pairs")

    for src, col in [('A (SHAP)', 'Source_A_ShapStable'), ('B (AGORA2)', 'Source_B_AGORA2'),
                     ('C (MOFA)', 'Source_C_MOFA')]:
        av = synth_df[f'{col}_Available']
        sup = synth_df[f'{col}_Support'] == True
        print(f"  Source {src}: available {int(av.sum())}/{len(synth_df)}, supporting {int(sup.sum())}")

    n_strong = int((synth_df['N_Sources_Supporting'] >= 3).sum())
    n_moderate = int((synth_df['N_Sources_Supporting'] == 2).sum())
    print(f"\nHigh-confidence pairs (all 3 sources support): {n_strong}")
    print(f"Moderate-confidence pairs (2 sources support): {n_moderate}")
    if n_strong + n_moderate > 0:
        print(synth_df.loc[synth_df['N_Sources_Supporting'] >= 2,
                            ['Species', 'Polyamine', 'Rho_partial', 'Evidence_Level']].to_string(index=False))


In [ ]:
if not synth_df.empty and synth_df['N_Sources_Supporting'].max() >= 2:
    high_conf = synth_df[synth_df['N_Sources_Supporting'] >= 2]
    pivot_e = high_conf.pivot_table(index='Species', columns='Polyamine',
                                     values='N_Sources_Supporting', fill_value=0, aggfunc='max')
    fig, ax = plt.subplots(figsize=(max(6, len(pivot_e.columns) * 2),
                                     max(5, len(pivot_e) * 0.35 + 2)))
    sns.heatmap(pivot_e, cmap='YlOrRd', vmin=0, vmax=3, annot=True, fmt='d',
                linewidths=0.4, ax=ax, cbar_kws={'label': 'N sources supporting (of 3)'})
    ax.set_title('Final Evidence Synthesis — partial-correlation candidates with >=2/3 sources supporting\n'
                  '(Source A: SHAP | Source B: AGORA2 enzyme | Source C: MOFA same-sign co-loading)',
                  fontsize=10, fontweight='bold')
    plt.tight_layout()
    out_synth = FIGURES / 'final_evidence_synthesis_heatmap_v2.png'
    fig.savefig(out_synth, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {out_synth}")
else:
    print("Synthesis figure skipped — no pairs with >=2/3 sources supporting.")


## Summary

Explicit availability flags (`HAS_GLASSO`, `HAS_GLASSO_NULL`, `HAS_MODULE_B`, `HAS_R1`,
`HAS_R8`, `HAS_MOFA_SCORES`, `HAS_MOFA_LOADINGS`, `HAS_PA_PRODUCERS`) are checked below
rather than relying on `try/except NameError`, which would also silently swallow genuine
bugs unrelated to a missing upstream input.


In [ ]:
print("=== 09 Pathway Network Analysis — Summary ===")

print("\nModule D (GLASSO + permutation null):")
if HAS_GLASSO:
    print(f"  Alpha selected: {glasso.alpha_:.4f}")
    print(f"  Edges saved: {len(glasso_edges)}")
    print(f"  Permutation-validated: {HAS_GLASSO_NULL}")
    print(f"  Permutation fits failed: {n_perm_failed}/{N_PERMUTATIONS} "
          "(counted as exceedances)")
    if HAS_GLASSO_NULL:
        print(f"  max-T permutation FWER-significant edges: {int(glasso_edges['Significant'].sum())}")
    print("  -> tables/d_glasso_partial_correlation_edges_v2.csv")
    print("  -> figures/pathway_network/d_glasso_network_v2.png")
else:
    print("  SKIPPED (GLASSO not numerically sound on this matrix; see module-d-fit)")

print("\nModule B (AGORA2 producer enzyme profile):")
if HAS_MODULE_B:
    print(f"  Producer species profiled: {len(b_profile_df)}")
    print(f"  With an AGORA2 model available: {int(b_profile_df['AGORA2_Model_Available'].sum())}")
    print("  -> tables/b_agora2_producer_enzyme_profile_v2.csv")
else:
    # The previous message blamed pa_producers even when the real reason was the
    # disabled genomic reference -- a skip recorded with the wrong reason.
    print("  SKIPPED (needs NB03 pa_producers AND a genomic reference index; "
          "the index is unavailable -- AGORA2 disabled)")

print("\nModule C (curated pathway mapping + species-pathway correlations):")
print(f"  Pathway-mapped metabolite features: {len(pathway_mapping_df)}")
print(f"  Significant species-pathway correlations (q<{utils.FDR_THRESHOLD}): {len(sig)}")
print("  -> tables/c_pathway_mapping_v2.csv")
print("  -> tables/c_species_pathway_correlations_v2.csv")
if not sig.empty:
    print("  -> figures/pathway_network/c_species_pathway_heatmap_v2.png")

print("\nModule G (pathway enrichment scores x MOFA+ factors, stage-stratified figure):")
print(f"  Pathway enrichment matrix: {enrich_df.shape}")
print("  -> tables/g_pathway_enrichment_scores_v2.csv")
print("  -> tables/g_pathway_factor_correlations_v2.csv")
if HAS_MOFA_SCORES:
    print("  -> figures/pathway_network/g_pathway_mofa_factor_heatmap_v2.png")
print("  -> figures/pathway_network/g_pathway_stage_boxplots_v2.png")

print("\nFinal synthesis (relocated cross-notebook evidence triangulation):")
print(f"  Candidate pairs scored: {len(synth_df)}")
if not synth_df.empty:
    print(f"  Pairs with all 3 sources available:  {int((synth_df['N_Sources_Available'] == 3).sum())}")
    print(f"  Pairs with 2 sources available:       {int((synth_df['N_Sources_Available'] == 2).sum())}")
    print(f"  Pairs with 1 source available:        {int((synth_df['N_Sources_Available'] == 1).sum())}")
    print(f"  Pairs with 0 sources available:       {int((synth_df['N_Sources_Available'] == 0).sum())}")
    print(f"  Pairs with all 3 sources supporting:  {int((synth_df['N_Sources_Supporting'] == 3).sum())}")
print("  -> tables/final_evidence_synthesis_v2.csv")
